#### Introduction
In this python notebbok, I'd like to recreate the experimentation of the DPO paper, specifically the summarization experiments wherein we'll use the reddit post summarization dataset as listed in the paper.

To do this, we need the following:
* Dataset(M. Völske et al "TL;DR: Mining Reddit to learn automatic summarization")
* SFT Model(Trained on the above dataset)
* Human preferances(N. Stiennon et al "Learning to summarize from human feedback")

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import PeftModel
import torch
from torch.utils.data import Dataset
import torch
import copy
from torch.optim.lr_scheduler import LinearLR
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use only GPU 0

/home/godwinkhalko/deformable-detr-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Dataset
Let's load the TLDR and the human preferances dataset

In [7]:
tldr_dataset = load_dataset("CarperAI/openai_summarize_tldr")

#### SFT Model
Let's train a SFT model on the TLDR Dataset and take it from there

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device)


In [ ]:

# Step 1: Format + Tokenize
def format_and_tokenize(example):
    result = tokenizer(
        example["prompt"] + example["label"],
        truncation=True,
        max_length=550,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = tldr_dataset.map(
    format_and_tokenize,
    remove_columns=tldr_dataset["train"].column_names,  # remove ALL original columns
    batched=False,
)

tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Step 2: Verify shapes before training
sample = tokenized_dataset["train"][:2]
for k, v in sample.items():
    print(f"{k}: {v.shape}, dtype: {v.dtype}")

In [ ]:
# def format_example(example):
#     return {"text": example["prompt"] + "\nTL;DR: " + example["label"]}

# tldr_dataset = tldr_dataset.map(format_example)

# trainer = SFTTrainer(
#     model=model,
#     train_dataset=tldr_dataset["train"],
#     eval_dataset=tldr_dataset["valid"],
#     dataset_text_field="text",
#     max_seq_length=550,
# )

In [ ]:
# Step 3: Train with standard Trainer
training_args = TrainingArguments(
    output_dir="./qwen-tldr-sft",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    logging_steps=100,
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    fp16=True,
    remove_unused_columns=False,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["valid"],
    data_collator=data_collator,
)

In [2]:
base_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
checkpoint_path = "./qwen-tldr-sft/checkpoint-43773"  # your checkpoint folder

# Load base + adapter
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="cpu"  # Merge on CPU to avoid OOM
)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

model = PeftModel.from_pretrained(model, checkpoint_path)

# Merge and unload adapter
model = model.merge_and_unload()

# Save merged model
model.save_pretrained("./qwen-tldr-sft-merged")
tokenizer.save_pretrained("./qwen-tldr-sft-merged")
print("Merged model saved!")

Merged model saved!


#### DPO Training 

In [3]:
# Load the preference dataset
preferences_dataset = load_dataset("openai/summarize_from_feedback", "comparisons")

In [4]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained("./qwen-tldr-sft-merged", torch_dtype=torch.float16).to(device)
tokenizer = AutoTokenizer.from_pretrained("./qwen-tldr-sft-merged") 


In [11]:
def summarise(post_text, tokenizer=tokenizer, model=model, device=device):
    text = post_text 
    tokens = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **tokens,
            do_sample=False,
            max_new_tokens=150,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded_tokens = output[0][tokens["input_ids"].shape[1]:]  # Get only the generated tokens
    summary = tokenizer.decode(decoded_tokens, skip_special_tokens=True)

    if "TL;DR" in summary:
        summary = summary.split("TL;DR")[0].strip()
    return summary

In [5]:
def get_logProbsRatio(text, dpo_model, sft_model,tokenizer, device):
        # Tokenize the prompt and label separately
        tokens_prompt = tokenizer(text["prompt"], return_tensors="pt",padding=True, truncation=True).to(device)
        tokens_label = tokenizer(text["completion"], return_tensors="pt",padding=True, truncation=True).to(device)

        # Concatenate the prompt and label tokens for the model input
        tokens = {
        "input_ids":      torch.cat([tokens_prompt["input_ids"],      tokens_label["input_ids"]],      dim=1),
        "attention_mask": torch.cat([tokens_prompt["attention_mask"],  tokens_label["attention_mask"]], dim=1),
        }

        prompt_len = tokens_prompt["input_ids"].shape[1]
        label_len = tokens_label["input_ids"].shape[1]
        # Get the model's output logits for the label tokens given the prompt tokens for both the SFT and DPO models
        sft_model.eval()
        dpo_model.eval()
        with torch.no_grad():
                output_sft = sft_model(**tokens)
        output_dpo = dpo_model(**tokens)

        # Extract the logits corresponding to the label tokens (i.e., the tokens after the prompt tokens)
        label_logits_sft = output_sft.logits[:, prompt_len - 1:prompt_len + label_len - 1, :]
        label_logits_dpo = output_dpo.logits[:, prompt_len - 1:prompt_len + label_len - 1, :]

        # Compute the probabilities of the label tokens for both models
        label_probs_sft = torch.log_softmax(label_logits_sft, dim=-1)
        label_probs_dpo = torch.log_softmax(label_logits_dpo, dim=-1)

        label_ids = tokens_label["input_ids"].unsqueeze(-1)
        label_probs_sft = label_probs_sft.gather(2, label_ids).squeeze(-1)
        label_probs_dpo = label_probs_dpo.gather(2, label_ids).squeeze(-1)

        #Compute the sequence probabilities by taking the product of the token probabilities
        label_mask = tokens_label["attention_mask"].bool()
        log_probs_sft = (label_probs_sft * label_mask).sum(dim=1)
        log_probs_dpo = (label_probs_dpo * label_mask).sum(dim=1)

        return log_probs_dpo - log_probs_sft 

In [3]:
#Since the preference dataset has a different structure, we need to format it similarly to the TL;DR dataset for training. The following function will help us achieve that.
def format_like_tldr(example):
    prompt = "SUBREDDIT: r/" + example["info"]["subreddit"] + " \nTITLE: " + example["info"]["title"] + " \nPOST: " + example["info"]["post"] + " \nTL;DR: "
    text1 = {"prompt": prompt, "completion": example["summaries"][0]["text"]}
    text2 = {"prompt": prompt, "completion": example["summaries"][1]["text"]}
    choice = example["choice"]
    return {"text1": text1, "text2": text2, "choice": choice}


In [4]:
#Let's create a Dataset from the preference dataset that is formatted like the TL;DR dataset. This will allow us to use the same training pipeline for both datasets.
class PreferenceDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        result_dict = format_like_tldr(example)
        return result_dict

In [8]:
def DPO_Loss(example, beta, dpo_model, sft_model, tokenizer, device):
    choice = example["choice"]
    log_probs_diff1 = get_logProbsRatio(example["text1"], dpo_model=dpo_model, sft_model=sft_model, tokenizer=tokenizer, device=device)
    log_probs_diff2 = get_logProbsRatio(example["text2"], dpo_model=dpo_model, sft_model=sft_model, tokenizer=tokenizer, device=device)
    sign = [1 if c == 0 else -1 for c in choice]
    loss = -torch.log(torch.sigmoid(beta * torch.tensor(sign).to(device) * (log_probs_diff1 - log_probs_diff2)))

    return loss.mean()


In [9]:
train_pref_dataset = PreferenceDataset(preferences_dataset["train"])
validation_pref_dataset = PreferenceDataset(preferences_dataset["validation"])

train_dataloader = torch.utils.data.DataLoader(train_pref_dataset, batch_size=4, shuffle=True)
validation_dataloader = torch.utils.data.DataLoader(validation_pref_dataset, batch_size=4)

In [ ]:
#Let's define the training loop for DPO. We will iterate through the preference dataset, compute the DPO loss for each example, and update the model parameters accordingly. 
# We will also evaluate the model on the validation set after each epoch to monitor its performance.
def train_dpo(sft_model, train_dataloader, validation_dataloader, tokenizer, device, beta=0.5, num_epochs=1):
    dpo_model = copy.deepcopy(sft_model).to(device)
    optimizer = torch.optim.RMSprop(dpo_model.parameters(), lr=1e-6)
    scheduler = LinearLR(
    optimizer,
    start_factor=1e-9,   # effectively starts near 0 (1e-9 * 1e-6 ≈ 0)
    end_factor=1.0,       # ends at full lr (1.0 * 1e-6 = 1e-6)
    total_iters=150
)
    sft_model.eval()  # Set SFT model to eval mode since it's not being updated
    for param in sft_model.parameters():
        param.requires_grad = False
        
    for epoch in range(num_epochs):
        dpo_model.train()
        total_loss = []
        for batch in train_dataloader:
            optimizer.zero_grad()
            loss = DPO_Loss(batch, beta, dpo_model, sft_model, tokenizer, device)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(dpo_model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            total_loss.append(loss.item())
            break

        avg_loss = sum(total_loss) / len(total_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {avg_loss:.4f}")

        # Validation loop
        dpo_model.eval()
        with torch.no_grad():
            val_loss = []
            for batch in validation_dataloader:
                loss = DPO_Loss(batch, beta, dpo_model, sft_model, tokenizer, device)
                val_loss.append(loss.item())
                break

        avg_val_loss = sum(val_loss) / len(val_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Validation Loss: {avg_val_loss:.4f}")
    # Free up memory after training
    del optimizer
    del scheduler

    return dpo_model

In [11]:
example = next(iter(train_dataloader))
loss = DPO_Loss(example, beta=0.5, dpo_model=model, sft_model= model, tokenizer=tokenizer, device=device)
print(f"Example DPO Loss: {loss.item():.4f}")

Example DPO Loss: 0.6931


In [ ]:
del model
# del dpo_model
del tokenizer
del train_dataloader
del validation_dataloader

: 

In [ ]:
sites = []
for example in preferences_dataset["validation"]:
    #if any of the values of the keys = {post, title, subreddit} is None, print the example
    if example["info"]["post"] is None or example["info"]["title"] is None or example["info"]["subreddit"] is None:
        broken_example = example
        break

In [7]:
#filter the validation dataset to only include examples from the site "None"
validation_dataset = [example for example in preferences_dataset["validation"] if example["info"]["site"] is None]

In [29]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
DPO_model = AutoModelForCausalLM.from_pretrained("/home/godwinkhalko/LLMs/qwen-tldr-dpo", torch_dtype=torch.float16).to(device)
SFT_model = AutoModelForCausalLM.from_pretrained("/home/godwinkhalko/LLMs/qwen-tldr-sft-merged_2", torch_dtype=torch.float16).to(device)
BASE_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", torch_dtype=torch.float16).to(device)
tokenizer_base = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
tokenizer_sft = AutoTokenizer.from_pretrained("/home/godwinkhalko/LLMs/qwen-tldr-sft-merged_2")
tokenizer_dpo= AutoTokenizer.from_pretrained("/home/godwinkhalko/LLMs/qwen-tldr-dpo")

OSError: Incorrect path_or_model_id: '/home/godwinkhalko/LLMs/qwen-tldr-dpo'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

predictions = []
references = []

for i in range(500):
    example = tldr_dataset["valid"][i]

    pred = summarise(
        example["prompt"],
        tokenizer_sft,
        SFT_model,
        device
    )

    predictions.append(pred)
    references.append(example["label"])

results = rouge.compute(
    predictions=predictions,
    references=references
)

print(results)

{'Rouge-L-P': 0.0016107237624972074, 'Rouge-L-R': 0.9853306859696522, 'Rouge-L-F': 0.9853306859696522}
{'Rouge-L-P': 0.0016146041669045268, 'Rouge-L-R': 0.984803563709358, 'Rouge-L-F': 0.984803563709358}


'SUBREDDIT: r/relationships\nTITLE: Me [19F] with my friend [19M] 10 months, Insecurities - Show or Tell?\nPOST: What are your stories about insecurities you\'ve had in past relationships? How have you dealt with them, particularly the ones that you can\'t hide?\n\nI\'m not currently in a relationship, but recently I\'ve realized that there is someone who likes me, and I\'m interested in them, too. Frankly, the only reason I\'m not asking them out is because I know that I have some insecurities that need to be worked through - particularly in the realm of body image. While I\'m confident in the rest of my body, I\'ve had terrible, awful acne both on my arms and breasts since I was very young. It\'s a special type with no complete cure, but doctors suggested that I keep my skin oiled until it goes away (dryness irritates it). Because of this it\'s not so much present anymore as large clusters of scars are.\n\nWould I warn someone about this upfront before anything sexual? Would I just l

In [ ]:
tldr_dataset["train"][100]["label"]

"My skin is scarred badly; what could I do/say about it that would gross my future partner out the least? What's your experience with body image issues?"

In [14]:
SFT_model = AutoModelForCausalLM.from_pretrained("/home/godwinkhalko/LLMs/qwen-tldr-sft-merged", torch_dtype=torch.float16).to(device)
tokenizer_sft = AutoTokenizer.from_pretrained("/home/godwinkhalko/LLMs/qwen-tldr-sft-merged")

index = 100
print("Prompt: ", tldr_dataset["train"][index]["prompt"])
print("Label: ", tldr_dataset["train"][index]["label"])
summarise(
    tldr_dataset["train"][index]["prompt"],
    tokenizer_sft,
    SFT_model,
    device
)

Prompt:  SUBREDDIT: r/relationships
TITLE: My Boyfriend [15m] Broke up with Me [17f] and is spreading Rumors about me. I don't want to break up, Help?
POST: I asked about this before but didnt really get any help :/ My boyfriend of 8 months broke up with me last weekend, and now he's spreading rumors about me around school. He's saying that I'm psycho/crazy and a bunch of stuff. What happened that led to the breakup is that some girl was texting him saying flirty stuff, he wasn't flirting back but she wasn't being appropriate at all and she knew he had a girlfriend.

I read some of these texts, I didn't go through his phone or anything. He left it in his room when he was in another and he got a text and I was going to bring it to him but it was from a girl so I just checked it. I didnt respond but I texted her from my phone and asked who she was and why she's texting my boyfriend and she told my boyfriend that i read their conversation and texted her.

Then he asked me about it and I j

'20 year old boyfriends broke up with me because of some girl talking to him, spread rumors about me on social media and now wants to date her'

In [13]:
import evaluate

rouge = evaluate.load("rouge")

predictions = []
references = []

for i in range(500):
    print(f"Processing example {i+1}/500", end="\r")
    example = tldr_dataset["valid"][i]

    pred = summarise(
        example["prompt"],
        tokenizer_sft,
        SFT_model,
        device
    )

    predictions.append(pred)
    references.append(example["label"])

results = rouge.compute(
    predictions=predictions,
    references=references
)

print(results)

{'rouge1': 0.2547851302885349, 'rouge2': 0.07821496785225995, 'rougeL': 0.1976258088250484, 'rougeLsum': 0.19763351685633201}


In [30]:
BASE_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", torch_dtype=torch.float16).to(device)
tokenizer_base = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
rouge = evaluate.load("rouge")

predictions = []
references = []

for i in range(100):
    print(f"Processing example {i+1}/100", end="\r")
    example = tldr_dataset["valid"][i]

    pred = summarise(
        example["prompt"],
        tokenizer_base,
        BASE_model,
        device
    )

    predictions.append(pred)
    references.append(example["label"])

results = rouge.compute(
    predictions=predictions,
    references=references
)

print(results)

/home/godwinkhalko/deformable-detr-env/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/godwinkhalko/deformable-detr-env/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/godwinkhalko/deformable-detr-env/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


{'rouge1': 0.19671355378763072, 'rouge2': 0.039495509086865624, 'rougeL': 0.1393829357182773, 'rougeLsum': 0.15205946547668817}
